# Notebook 06: Flash Attention

## Why This Matters
Standard attention materializes the full N×N attention matrix in GPU HBM (high-bandwidth memory). At sequence length 4096 with 32 heads, that's ~2 GB of intermediate tensors — just for attention weights. Flash Attention (Dao et al., 2022) computes the exact same result without ever writing the full attention matrix to memory, cutting attention memory from O(N²) to O(N) and running 2–4× faster in practice.

This was the algorithmic breakthrough that made 100k+ context windows practical. Every modern inference stack (vLLM, TGI, llama.cpp, PyTorch's `scaled_dot_product_attention`) uses it.

## Structure
1. The memory bottleneck in standard attention
2. GPU memory hierarchy — why HBM bandwidth is the constraint
3. The tiling idea — compute attention in blocks
4. Online softmax — the key algorithmic trick
5. Flash Attention forward pass
6. Flash Attention 2 improvements
7. Benchmarking: standard vs Flash Attention

## What You'll Learn
- Why memory bandwidth, not FLOPs, is the bottleneck for attention
- How the online softmax trick enables block-wise computation
- The exact memory savings at different sequence lengths
- Why Flash Attention is IO-bound not compute-bound

## Background

### The problem Flash Attention solves

Standard attention computes three intermediate tensors before producing output: the raw scores matrix S (Q×Kᵀ), the softmax probabilities P, and then the output O (P×V). At sequence length N with H attention heads, S and P each have shape (H, N, N). At N=4096 with 32 heads in float32, that's 4 GB of intermediate tensors — just for one attention layer.

The problem isn't the FLOPs. GPUs are extremely fast at matrix multiplication. The problem is that S and P must be *written to GPU main memory (HBM) and read back* for each step. HBM bandwidth is the bottleneck, not compute.

### GPU memory hierarchy: why this matters

Modern GPUs have a two-tier memory hierarchy. **SRAM** (on-chip registers and L1 cache) is extremely fast — ~19 TB/s on an A100 — but tiny, around 192 KB per streaming multiprocessor. **HBM** (high-bandwidth memory, the GPU's main memory) is ~2 TB/s — about 9× slower than SRAM, but holds the actual tensors (tens of GB).

Standard attention is *memory-bandwidth bound*: the bottleneck isn't the matrix multiplications (which run on tensor cores at high utilization), but the repeated HBM reads and writes of the N×N attention matrices. Making attention faster required reducing HBM traffic, not reducing FLOPs.

### The Flash Attention insight (Dao et al., 2022)

The key observation: we don't need to store the full N×N matrix. We just need the output O, which is a weighted sum of V. If we process attention in **tiles** (small blocks that fit in SRAM), we can compute O incrementally — accumulating partial results block by block — and never write the full N×N matrix to HBM at all.

The obstacle: softmax requires knowing all scores to normalize. If we only see a tile, we don't know the denominator. The solution is **online softmax** — a numerically stable algorithm from Milakov & Gimelshein (2018) that computes softmax incrementally with running corrections. As each new tile is processed, previous partial results are corrected using the updated running maximum and sum.

Flash Attention 1 (June 2022) demonstrated 2–4× speedup over standard attention and reduced memory from O(N²) to O(N), making 100k+ context windows practical for the first time. Flash Attention 2 (2023) added better parallelism across the sequence dimension and reduced non-matmul overhead, achieving ~70% of theoretical GPU FLOP utilization.

### Impact

Flash Attention was one of the most consequential ML engineering papers in recent years. Within months of publication, it was integrated into PyTorch (`F.scaled_dot_product_attention`), HuggingFace Transformers, vLLM, and virtually every production LLM serving stack. Models that previously had context limits of 2k–4k tokens could now handle 32k–100k with the same hardware budget. Long-context LLMs (Claude 2's 100k window, GPT-4's 128k window) are direct descendants of this work.

In [ ]:
%pip install -q torch

In [ ]:
import math
import time
import torch
import torch.nn.functional as F

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. The Memory Bottleneck in Standard Attention

Standard attention computes:
```
S = QK^T / sqrt(d)     shape: (batch, heads, N, N)   ← written to HBM
P = softmax(S)          shape: (batch, heads, N, N)   ← written to HBM  
O = PV                  shape: (batch, heads, N, d)   ← written to HBM
```

Each of those intermediate tensors is read back from HBM for the next step. At N=4096, heads=32, d=128:

```
S tensor: 32 × 4096 × 4096 × 4 bytes = 2.0 GB
P tensor: 32 × 4096 × 4096 × 4 bytes = 2.0 GB
O tensor: 32 × 4096 × 128  × 4 bytes = 0.06 GB

HBM reads + writes: ~8 GB total just for attention
```

Flash Attention avoids materializing S and P entirely — it tiles the computation so that the working set fits in SRAM (on-chip cache, ~100× faster than HBM).

In [ ]:
def attention_memory_gb(batch: int, n_heads: int, seq_len: int, d_head: int, dtype_bytes: int = 2) -> dict:
    """Memory usage for standard attention intermediate tensors."""
    N = seq_len
    # S = QK^T: (batch, heads, N, N)
    s_bytes = batch * n_heads * N * N * dtype_bytes
    # P = softmax(S): same shape
    p_bytes = s_bytes
    # Q, K, V: (batch, heads, N, d_head) each
    qkv_bytes = 3 * batch * n_heads * N * d_head * dtype_bytes
    # Output O: (batch, heads, N, d_head)
    o_bytes = batch * n_heads * N * d_head * dtype_bytes

    return {
        "S (attn scores)": s_bytes / 1e9,
        "P (attn weights)": p_bytes / 1e9,
        "Q+K+V": qkv_bytes / 1e9,
        "O (output)": o_bytes / 1e9,
        "Total HBM": (s_bytes + p_bytes + qkv_bytes + o_bytes) / 1e9,
    }


print("Standard attention memory (BF16, batch=1, n_heads=32, d_head=128):")
print(f"{'Seq len':>10} | {'S (GB)':>8} | {'P (GB)':>8} | {'Q+K+V (GB)':>12} | {'Total (GB)':>12}")
print("-" * 60)
for N in [512, 1024, 2048, 4096, 8192, 16384]:
    mem = attention_memory_gb(1, 32, N, 128)
    print(f"{N:>10} | {mem['S (attn scores)']:>8.3f} | {mem['P (attn weights)']:>8.3f} | "
          f"{mem['Q+K+V']:>12.3f} | {mem['Total HBM']:>12.3f}")

print()
print("Flash Attention memory: O(N×d) only — keeps only tiles in SRAM, no N×N tensors in HBM")
print(f"Flash at N=16384: ~{1 * 32 * 16384 * 128 * 2 * 4 / 1e9:.3f} GB (Q+K+V+O only)")

## 2. GPU Memory Hierarchy

To understand why Flash Attention matters, you need to understand where computation happens:

```
┌─────────────────────────────────────────────────────────┐
│  GPU Computing Units (CUDA cores / tensor cores)        │
│  - This is where matrix multiplications happen          │
│  - Extremely fast: ~300 TFLOPS (A100 BF16)              │
└──────────────────┬──────────────────────────────────────┘
                   │ feeds
┌──────────────────▼──────────────────────────────────────┐
│  SRAM (shared memory / L1 cache)  ~192 KB per SM        │
│  - On-chip, extremely fast: ~19 TB/s                    │
│  - Very small — can only hold ~8 tiles at once          │
└──────────────────┬──────────────────────────────────────┘
                   │ reads/writes (SLOW)
┌──────────────────▼──────────────────────────────────────┐
│  HBM (High Bandwidth Memory)  40–80 GB                  │
│  - Off-chip, much slower: ~2 TB/s (vs 19 TB/s SRAM)     │
│  - Where model weights, KV cache, and activations live  │
└─────────────────────────────────────────────────────────┘

Key ratio: SRAM is ~9.5× faster than HBM
Standard attention: repeatedly reads/writes N×N matrix to HBM
Flash Attention: keeps tiles in SRAM, only reads/writes O(N) to HBM
```

Standard attention is **memory-bandwidth bound** (HBM reads/writes dominate), not compute-bound. Even if FLOPs don't change, reducing HBM traffic directly reduces runtime.

## 3. The Tiling Idea

Flash Attention computes attention in **tiles** (blocks). For a tile of Q rows and a tile of K/V columns:

```
Full N×N attention:          Tiled attention:

Q ──────► K^T               Q_block ──► K_block^T
  (N×d)   (d×N)              (Br×d)    (d×Bc)
     ↓                           ↓
    S (N×N)                 S_block (Br×Bc)   ← fits in SRAM!
     ↓                           ↓
  softmax                  partial softmax    ← needs correction
     ↓                           ↓
   P×V → O                 O_block            ← accumulate
```

**The problem:** Softmax requires seeing ALL scores to normalize. If we only see a tile, we can't compute the correct denominator.

**The solution:** Online softmax — compute a running correction as new tiles are processed.

## 4. Online Softmax — The Key Trick

Standard softmax: $p_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$

To compute this stably, we subtract the max: $p_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}$ where $m = \max_j x_j$

**Online update rule:** When we process a new block and discover new values, we can update our running statistics:

```
After block 1: m_1 = max(scores_1),  l_1 = sum(exp(scores_1 - m_1))
After block 2: m_2 = max(m_1, max(scores_2))
               l_2 = l_1 × exp(m_1 - m_2) + sum(exp(scores_2 - m_2))
               O_corrected = O_1 × exp(m_1 - m_2) / l_2 × l_1 + O_2 / l_2
```

This allows us to process blocks sequentially and correct the output as we go — no need to store the full N×N matrix.

In [ ]:
def online_softmax_demo():
    """Demonstrate that online softmax produces identical results to standard softmax."""
    torch.manual_seed(42)
    x = torch.randn(16)  # 16 scores

    # Standard softmax
    standard = F.softmax(x, dim=-1)

    # Online softmax: process in blocks of 4
    block_size = 4
    m = float('-inf')  # running max
    l = 0.0            # running sum of exp
    online = torch.zeros(16)

    blocks = x.reshape(-1, block_size)
    block_outputs = []

    for b_idx, block in enumerate(blocks):
        m_new = max(m, block.max().item())
        l_new = l * math.exp(m - m_new) + torch.exp(block - m_new).sum().item()
        block_exp = torch.exp(block - m_new)

        # Correct previous output
        for prev_b in range(b_idx):
            start = prev_b * block_size
            online[start:start+block_size] *= (l / l_new) * math.exp(m - m_new)

        # Add new block
        start = b_idx * block_size
        online[start:start+block_size] = block_exp / l_new

        m, l = m_new, l_new

    print("Online softmax demo:")
    print(f"Standard softmax: {standard[:8].numpy().round(4)}")
    print(f"Online softmax:   {online[:8].numpy().round(4)}")
    print(f"Max absolute diff: {(standard - online).abs().max().item():.2e}")
    print(f"Sums to 1: {online.sum().item():.6f}")

online_softmax_demo()

## 5. Flash Attention Forward Pass

The full Flash Attention algorithm. This is a Python implementation for understanding — the real implementation is a CUDA kernel.

In [ ]:
def flash_attention_forward(
    Q: torch.Tensor,  # (N, d)
    K: torch.Tensor,  # (N, d)
    V: torch.Tensor,  # (N, d)
    block_size: int = 64,
    causal: bool = True,
) -> torch.Tensor:
    """
    Flash Attention forward pass — Python reference implementation.
    Computes exact same result as standard attention without materializing
    the full N×N attention matrix.

    In practice this is a CUDA kernel, not Python. This is for understanding only.
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    Br = Bc = block_size  # row and column block sizes

    O = torch.zeros(N, d, dtype=Q.dtype)  # output accumulator
    l = torch.zeros(N)                     # normalizer (sum of exp)
    m = torch.full((N,), float('-inf'))    # running max

    # Iterate over column blocks (K, V blocks)
    for j_start in range(0, N, Bc):
        j_end = min(j_start + Bc, N)
        K_j = K[j_start:j_end]  # (Bc, d)
        V_j = V[j_start:j_end]  # (Bc, d)

        # Iterate over row blocks (Q blocks)
        for i_start in range(0, N, Br):
            i_end = min(i_start + Br, N)

            # Causal masking: skip blocks where all queries are before all keys
            if causal and j_start > i_end:
                continue

            Q_i = Q[i_start:i_end]   # (Br, d)
            O_i = O[i_start:i_end]   # (Br, d)
            l_i = l[i_start:i_end]   # (Br,)
            m_i = m[i_start:i_end]   # (Br,)

            # Compute attention scores for this block
            S_ij = (Q_i @ K_j.T) * scale  # (Br, Bc)

            # Apply causal mask within the block
            if causal:
                row_idx = torch.arange(i_start, i_end).unsqueeze(1)
                col_idx = torch.arange(j_start, j_end).unsqueeze(0)
                S_ij = S_ij.masked_fill(col_idx > row_idx, float('-inf'))

            # Online softmax update
            m_ij = S_ij.max(dim=-1).values          # (Br,) new block max
            m_new = torch.maximum(m_i, m_ij)         # (Br,) updated running max

            P_ij = torch.exp(S_ij - m_new.unsqueeze(1))  # (Br, Bc) stable exp
            l_ij = P_ij.sum(dim=-1)                       # (Br,)

            # Correction factor for previously accumulated output
            correction = torch.exp(m_i - m_new)  # (Br,)

            # Update normalizer and output
            l_new = correction * l_i + l_ij
            O_new = (correction.unsqueeze(1) * l_i.unsqueeze(1) * O_i +
                     P_ij @ V_j) / l_new.unsqueeze(1)

            # Write back
            O[i_start:i_end] = O_new
            l[i_start:i_end] = l_new
            m[i_start:i_end] = m_new

    return O


def standard_attention(Q, K, V, causal=True):
    """Standard attention for comparison."""
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    S = (Q @ K.T) * scale
    if causal:
        mask = torch.tril(torch.ones(N, N, dtype=torch.bool))
        S = S.masked_fill(~mask, float('-inf'))
    P = F.softmax(S, dim=-1)
    return P @ V


# Verify Flash Attention produces identical results
torch.manual_seed(0)
N, d = 128, 64
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

out_standard = standard_attention(Q, K, V)
out_flash = flash_attention_forward(Q, K, V, block_size=32)

max_diff = (out_standard - out_flash).abs().max().item()
print(f"Standard attention output shape: {out_standard.shape}")
print(f"Flash attention output shape:    {out_flash.shape}")
print(f"Max absolute difference:         {max_diff:.2e}")
print(f"Outputs match (tol=1e-4):        {max_diff < 1e-4}")

## 6. Memory Complexity Analysis

In [ ]:
import sys

def measure_peak_memory_standard(N: int, d: int) -> float:
    """Estimate peak memory for standard attention (theoretical)."""
    # S and P matrices dominate: 2 × N × N × 4 bytes (float32)
    return 2 * N * N * 4 / 1e6  # MB


def measure_peak_memory_flash(N: int, d: int, block_size: int = 64) -> float:
    """Estimate peak memory for Flash Attention (theoretical)."""
    # Only O, l, m (all N × d or N): O(N × d)
    # Plus working tiles: 2 × block_size × d (Q_i, K_j/V_j tiles in SRAM)
    o_mem = N * d * 4  # output accumulator
    lm_mem = 2 * N * 4  # l and m vectors
    sram_tiles = 3 * block_size * d * 4  # Q, K, V tiles
    return (o_mem + lm_mem + sram_tiles) / 1e6  # MB


print("Memory complexity comparison (float32, d=64):")
print(f"{'N':>8} | {'Standard (MB)':>15} | {'Flash (MB)':>12} | {'Ratio':>8}")
print("-" * 50)
for N in [256, 512, 1024, 2048, 4096, 8192, 16384]:
    std = measure_peak_memory_standard(N, 64)
    flash = measure_peak_memory_flash(N, 64)
    ratio = std / flash
    print(f"{N:>8} | {std:>15.1f} | {flash:>12.1f} | {ratio:>7.0f}×")

## 7. PyTorch's Built-in Flash Attention

PyTorch 2.0+ includes `F.scaled_dot_product_attention` which automatically selects Flash Attention when available (CUDA with correct dtype, or MPS on Apple Silicon).

In [ ]:
import torch

# Check Flash Attention availability
print("Flash Attention backends available:")
print(f"  Flash Attention (CUDA):  {torch.backends.cuda.flash_sdp_enabled() if torch.cuda.is_available() else 'N/A (no CUDA)'}")
print(f"  Memory-efficient SDPA:   {torch.backends.cuda.mem_efficient_sdp_enabled() if torch.cuda.is_available() else 'N/A (no CUDA)'}")
print(f"  Math SDPA (fallback):    True (always available)")
print(f"  MPS device:              {torch.backends.mps.is_available()}")
print()

# Use scaled_dot_product_attention (FA under the hood when possible)
batch, heads, N, d = 2, 8, 512, 64
Q = torch.randn(batch, heads, N, d, device=device)
K = torch.randn(batch, heads, N, d, device=device)
V = torch.randn(batch, heads, N, d, device=device)

# With causal mask — the standard for decoder-only models
with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=True, enable_mem_efficient=True) if torch.cuda.is_available() else torch.no_grad():
    out_sdpa = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

print(f"scaled_dot_product_attention output: {out_sdpa.shape}")
print(f"Device: {out_sdpa.device}")

# Compare against manual standard attention
def manual_causal_attention(Q, K, V):
    scale = Q.shape[-1] ** -0.5
    S = (Q @ K.transpose(-2, -1)) * scale
    mask = torch.tril(torch.ones(N, N, dtype=torch.bool, device=Q.device))
    S = S.masked_fill(~mask, float('-inf'))
    return F.softmax(S, dim=-1) @ V

out_manual = manual_causal_attention(Q, K, V)
diff = (out_sdpa - out_manual).abs().max().item()
print(f"\nMax diff vs manual attention: {diff:.2e}  (numerical precision differences are expected)")

## 8. Benchmarking: Standard vs Flash Attention

In [ ]:
def benchmark_attention(seq_lens, batch=1, heads=32, d_head=64, n_trials=3):
    """Benchmark standard vs SDPA attention at various sequence lengths."""
    results = []

    for N in seq_lens:
        Q = torch.randn(batch, heads, N, d_head, device=device, dtype=torch.float32)
        K = torch.randn(batch, heads, N, d_head, device=device, dtype=torch.float32)
        V = torch.randn(batch, heads, N, d_head, device=device, dtype=torch.float32)

        # Warm up
        for _ in range(2):
            _ = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

        # Standard attention timing
        times_std = []
        for _ in range(n_trials):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            S = (Q @ K.transpose(-2, -1)) / math.sqrt(d_head)
            mask = torch.tril(torch.ones(N, N, dtype=torch.bool, device=device))
            S = S.masked_fill(~mask, float('-inf'))
            P = F.softmax(S, dim=-1)
            O = P @ V
            if device.type == "cuda":
                torch.cuda.synchronize()
            times_std.append(time.perf_counter() - t0)
        _ = O  # prevent optimization away

        # SDPA (Flash Attention) timing
        times_sdpa = []
        for _ in range(n_trials):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            O_sdpa = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times_sdpa.append(time.perf_counter() - t0)

        std_ms = min(times_std) * 1000
        sdpa_ms = min(times_sdpa) * 1000
        attn_mem_mb = 2 * batch * heads * N * N * 4 / 1e6  # S + P matrices
        results.append((N, std_ms, sdpa_ms, attn_mem_mb))

    return results


seq_lens = [128, 256, 512, 1024]
print(f"Benchmarking on {device}:")
print(f"{'N':>8} | {'Standard (ms)':>14} | {'SDPA (ms)':>12} | {'Speedup':>9} | {'Attn matrix (MB)':>18}")
print("-" * 70)

results = benchmark_attention(seq_lens)
for N, std_ms, sdpa_ms, mem_mb in results:
    speedup = std_ms / sdpa_ms
    print(f"{N:>8} | {std_ms:>14.2f} | {sdpa_ms:>12.2f} | {speedup:>8.2f}× | {mem_mb:>18.1f}")

print()
print("Note: On CUDA, Flash Attention speedup is 2–4× at large N.")
print("On CPU/MPS, SDPA still avoids materializing the full attention matrix.")

## 9. Flash Attention 2 Improvements

Flash Attention 2 (Dao, 2023) improves on FA1 with:

**1. Fewer non-matmul FLOPs**
FA1 has significant overhead from the online softmax correction. FA2 restructures the loop to minimize non-matmul operations (which are ~16× slower than matmuls on tensor cores).

**2. Better parallelism**
FA1 parallelizes across batch × heads only. FA2 also parallelizes across the sequence length dimension, better utilizing all CUDA threads.

**3. Better work partitioning**
FA2 splits work between threads more efficiently for both forward and backward passes.

**Result:** FA2 is ~2× faster than FA1, and achieves ~70% of theoretical GPU peak FLOP utilization.

In [ ]:
# IO complexity summary — the key metric for understanding Flash Attention
print("IO complexity (HBM reads + writes):")
print()

configs = [
    ("N=512, d=64",   512,  64),
    ("N=1024, d=64", 1024,  64),
    ("N=2048, d=128", 2048, 128),
    ("N=4096, d=128", 4096, 128),
]

B = 64  # SRAM block size (bytes), simplified
print(f"{'Config':<18} | {'Standard IO':>14} | {'Flash IO':>12} | {'IO Reduction':>13}")
print("-" * 65)
for name, N, d in configs:
    # Standard: read Q,K,V + write S + read S + write P + read P,V + write O
    # ≈ 4Nd + 2N² (dominant term is N²)
    standard_io = (4 * N * d + 2 * N * N) * 4  # bytes, float32

    # Flash: read Q,K,V + write O + small overhead
    # ≈ 4Nd (no N² term)
    flash_io = 4 * N * d * 4  # bytes

    reduction = standard_io / flash_io
    print(f"{name:<18} | {standard_io/1e6:>12.1f}MB | {flash_io/1e6:>10.1f}MB | {reduction:>12.0f}×")

## Summary

```
Standard Attention:
  Memory: O(N²)  — writes full attention matrix to HBM
  IO:     O(N²)  — repeatedly reads/writes N×N tensors
  Limit:  HBM bandwidth bound for large N

Flash Attention:
  Memory: O(N)   — only O+l+m vectors in HBM, tiles in SRAM
  IO:     O(N)   — reads Q,K,V once, writes O once
  Trick:  Online softmax allows block-wise computation
  Result: Exact same output, 2-4× faster, enables N=100k+
```

**Key insight:** Flash Attention doesn't reduce FLOPs — it reduces memory bandwidth usage. The attention matrix was never the compute bottleneck; it was the HBM traffic.

**In practice:** Use `F.scaled_dot_product_attention(Q, K, V, is_causal=True)` in PyTorch. It automatically dispatches to Flash Attention when running on supported hardware.

**Next:** Notebook 07 covers PagedAttention — vLLM's memory management system that solves KV cache fragmentation for concurrent serving.